# Cross-validation sweep — BanglaPoliticalStance

Runs every model in `configs/` under **one** protocol: grouped stratified 5-fold CV,
augmentation inside training folds, out-of-fold predictions pooled.

**Setup:** Runtime → Change runtime type → **T4 GPU**

**6 cells, run them in order. No tokens needed.**

In [ ]:
# Cell 1: GPU check
!nvidia-smi -L
import torch; print(f'torch {torch.__version__}, CUDA {torch.cuda.is_available()}')

In [ ]:
# Cell 2: Clone + install (repo is public)
import os, subprocess
if os.path.exists('/content/bangla-multimodal-political-stance'):
    !rm -rf /content/bangla-multimodal-political-stance

!git clone --depth 1 https://github.com/kishormorol/bangla-multimodal-political-stance.git /content/bangla-multimodal-political-stance
%cd /content/bangla-multimodal-political-stance
!pip install -q -e . 2>&1 | tail -3
!pip install -q gdown datasets 2>&1 | tail -1
!python -c "import bmpb; print('bmpb installed OK')"

In [ ]:
# Cell 3: Build corpus.csv (try Drive first, fall back to HF dataset)
%cd /content/bangla-multimodal-political-stance
import subprocess, sys, time
from pathlib import Path

# Try downloading from Drive (with retries)
for attempt in range(1, 4):
    print(f'
=== bmpb data attempt {attempt}/3 ===')
    subprocess.run([sys.executable, '-m', 'bmpb.cli', 'data'])
    csvs = list(Path('data/raw').glob('*.csv'))
    if len(csvs) >= 20:
        print(f'Got {len(csvs)} CSVs.')
        break
    print(f'Got {len(csvs)} CSVs, retrying in 20s...')
    time.sleep(20)

# Try bmpb ingest
result = subprocess.run([sys.executable, '-m', 'bmpb.cli', 'ingest'],
                        capture_output=True, text=True)
print(result.stdout)

# Fallback: build corpus.csv from HF if ingest failed
corpus_path = Path('data/processed/corpus.csv')
if not corpus_path.exists():
    print('
Ingest failed — building corpus.csv + downloading images from HF dataset...')
    from datasets import load_dataset
    import pandas as pd

    # Load with decode=True (default) so `image` is a PIL Image object when present.
    # The annotated split has 198 rows; images are ~8 MB total, fits in Colab RAM easily.
    ds = load_dataset('kishormorol/BanglaPoliticalStance', split='annotated')
    LABEL_NAMES = {0: 'govt_critique', 1: 'neutral', 2: 'govt_leaning'}

    # Save images to data/raw/processed_images/{item_id}.jpg
    # (matches IMAGES_PROCESSED in src/bmpb/paths.py)
    images_dir = Path('data/raw/processed_images')
    images_dir.mkdir(parents=True, exist_ok=True)

    rows = []
    n_saved = 0
    for item in ds:
        item_id = item['item_id']
        pil_img = item['image']  # PIL Image or None

        if pil_img is not None:
            img_filename = f'{item_id}.jpg'
            img_save_path = images_dir / img_filename
            # Convert to RGB before saving as JPEG (avoids RGBA/palette errors)
            pil_img.convert('RGB').save(img_save_path, format='JPEG', quality=95)
            # Relative path from repo root, as crossval.py reads corpus.csv
            image_path = f'data/raw/processed_images/{img_filename}'
            has_image = True
            n_saved += 1
        else:
            image_path = ''
            has_image = False

        rows.append({
            'item_id': item_id,
            'title': item['headline'],
            'text': item['headline'],
            'label': item['label'],
            'label_name': LABEL_NAMES[item['label']],
            'article_label_name': '',
            'image_label_name': '',
            'outlet': item['outlet'],
            'outlet_key': item['outlet'].lower(),
            'date': item['date'],
            'source_url': item['source_url'],
            'image_url': '',
            'image_path': image_path,
            'image_kind': 'photo' if has_image else '',
            'has_image': has_image,
            'annotator_1': '',
            'annotator_2': '',
            'annotator_3': '',
            'article_label': '',
            'image_label': '',
            'text_level': 'headline',
            'source_index': item_id,
        })

    corpus = pd.DataFrame(rows)
    corpus_path.parent.mkdir(parents=True, exist_ok=True)
    corpus.to_csv(corpus_path, index=False)
    print(f'Created corpus.csv: {len(corpus)} items from HF ({n_saved} with images)')
    print(f'Images saved to: {images_dir}')

# Verify
import pandas as pd
df = pd.read_csv(corpus_path)
print(f'
Corpus ready: {len(df)} items')
print(f'Labels: {df["label_name"].value_counts().to_dict()}')
print(f'Items with images: {df["has_image"].sum()}')

In [ ]:
# Cell 4: Run text model CV sweep
%cd /content/bangla-multimodal-political-stance
!PYTHON=$(which python) bash scripts/run_cv.sh configs/text

In [ ]:
# Cell 5: Run multimodal model CV sweep
%cd /content/bangla-multimodal-political-stance
!PYTHON=$(which python) bash scripts/run_cv.sh configs/multimodal

In [ ]:
# Cell 6: Leaderboard + download
%cd /content/bangla-multimodal-political-stance
!python -m bmpb.cli leaderboard
print(open('reports/tables/leaderboard.md').read())

!tar czf /content/cv-runs.tar.gz experiments reports
from google.colab import files
files.download('/content/cv-runs.tar.gz')